In [ ]:
puts `ls -l`

In [14]:
SOURCE='./raw_data/bv-kg-20260617.large'.freeze

(irb): warning: already initialized constant Object::SOURCE


"./raw_data/bv-kg-20260617.large"

In [ ]:
puts `head -2 #{SOURCE}`

In [ ]:
require 'csv'
drugs = {}
sourcesleft = {}
sourcesright = {}
CSV.foreach(SOURCE, col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|
  sourcesleft[row["type_1"]] = 1
  sourcesright[row["type_2"]] = 1
end
puts sourcesleft.keys
puts
puts sourcesright.keys


In [ ]:
puts `head -5 ./maps/2026-biovista-drugs.map`

# genes have two sources

biovista-genes  are from NCBI and UniProt

and

biovista-mesh (these are mostly gene categories not genes...)

Handled separately

In [ ]:
puts `head -5 ./maps/2025-biovista-genes.map`

In [15]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'

graphing_errors = File.open('./graph/2026_drug-gene-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
drug_mappings = CSV.read('./maps/2026-biovista-drugs.map', headers: true)
gene_mappings = CSV.read('./maps/2025-biovista-genes.map', headers: true)


failures = {}
# Process each entity relation

# refresh
f = File.open('./graph/2026_biovista_drug-gene.nq.large', 'w')
f.close
    
recordcount = 0
CSV.foreach(SOURCE, col_sep: "\t", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
    
        # Disease
        # Pathway
        # Drug
        # Human Phenotype
        # Gene
  next unless ["Drug", "Compound"].include?(row['type_1']) or ["Drug", "Compound"].include?(row['type_2'])
  next unless row['type_1'] == "Gene" or row['type_2'] == "Gene"
  if ["Drug", "Compound"].include?(row['type_1'])
    drug_id = row['id_1']
    gene_id = row['id_2']  # gene ids from NCBI are only numerical, from MeSH have a letter
  else
    drug_id = row['id_2']
    gene_id = row['id_1']
  end

  score = row['score']
  evidence = row['url']
    
  # Find corresponding mappings
  
  # mesh is going to be differnt from NCBI
  next if gene_id =~ /[A-Z]/  # this is a MeSH id, so we can't deal with it here

    recordcount = recordcount + 1
    warn "drug #{drug_id} gene #{gene_id}" if (recordcount) % 5000 == 0  # log every 5000 records because the notebook gets overloaded

#   warn "searching for #{drug_id}"
# biovista_meshid,biovista_label,CID,IUPACname
# http://purl.bioontology.org/ontology/MESH/D005680,gamma-Aminobutyric Acid,119,gamma-Aminobutyric Acid
  # Find corresponding mappings
  drug = drug_mappings.find { |d| d['biovista_meshid'] =~ /\/#{drug_id}/ }

  gene = gene_mappings.find { |d| d['bv_geneid'] == gene_id }
  
#     warn "found #{drug}"
  unless drug
    next if failures[drug_id]
    failures[drug_id] = 1
    warn "drug lookup failed #{drug_id}"
    graphing_errors.write "drug lookup failed #{drug_id}\n"
    next
  end
  unless gene
    next if failures[gene_id]
    failures[gene_id] = 1
    warn "gene lookup failed #{gene_id}"
    graphing_errors.write "gene lookup failed #{gene_id}\n"
    next
  end
  
  # Extract relevant IDs and labels
  # biovista_meshid,biovista_label,CID,IUPACname
  # http://purl.bioontology.org/ontology/MESH/D005680,gamma-Aminobutyric Acid,119,4-aminobutanoic acid
  drug_cid = drug['CID']
  if drug_cid =~ /SUBSTANCE_(\d+)/
    pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/substance/#{$1}")
  else
    pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/compound/#{drug_cid}")
  end
  pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
  pubchem_label =  RDF::Literal.new("PubChem Identifier")
  pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
  iupac_drug_label = RDF::Literal.new(drug['IUPACname'])

  #   bv_geneid,bv_label,geneid,protein,recommended_full,taxon
  # 11758,GPx,http://purl.uniprot.org/geneid/11758,http://purl.uniprot.org/uniprot/D3Z0Y2,Peroxiredoxin-6,http://purl.uniprot.org/taxonomy/10090
  gene_uri = RDF::URI.new(gene['geneid'])
  gene_type = RDF::URI.new("http://edamontology.org/data_1027")
  gene_label =  RDF::Literal.new("NCBI/UniProt Gene Identifier")
  gene_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Gene")
  biovista_gene_label = RDF::Literal.new(gene['recommended_full'])

  
  protein_uri = RDF::URI.new(gene['protein'])
  protein_type = RDF::URI.new("http://edamontology.org/data_2291")
  protein_label =  RDF::Literal.new("UniProt Identifier")
  protein_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Protein")
  biovista_protein_label = RDF::Literal.new(gene['recommended_full'])
    
  taxon = RDF::URI.new(gene['taxon'])
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:bv_#{drug_id}_#{gene_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")

  # Create RDF repository (need to do this each time, since there are hundreds of thousands of lines, and the graph gets too big for memory)
  graph = RDF::Repository.new
    
  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'],gene_uri , graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
  
  graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     iupac_drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_type,          graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_core_type,             graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_type, RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_core_type, RDFS.label,     RDF::Literal.new("Drug"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{drug_id}"), graph_name: context_uri)
  
  graph << RDF::Statement.new(gene_uri,  RDFS.label,       biovista_gene_label , graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_type, RDFS.label,       RDF::Literal.new("NCBI Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_core_type, RDFS.label,  RDF::Literal.new("Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)
  

  graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], protein_uri, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri, SIMPATHIC['associated-with'], pubchem_uri , graph_name: context_uri)
  
  graph << RDF::Statement.new(protein_uri,  RDFS.label,       biovista_protein_label , graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_type, RDFS.label,       RDF::Literal.new("UniProt"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_core_type, RDFS.label,  RDF::Literal.new("Protein"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)

  
  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Biovista"),  graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'], RDF::URI.new(evidence),  graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['score'], RDF::Literal.new(score),  graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'], RDF::Literal.new("ASSOCIATED_WITH"),  graph_name: general_context)


#   warn "graph #{context_uri} built"
  # Write RDF to file in N-Quads format
  File.open('./graph/2026_biovista_drug-gene.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
      writer << graph
    end
  end
#   warn "end graph writing"

end
warn "completed graph building"
graphing_errors.close

puts "RDF quads written"


(irb):7: warning: already initialized constant Object::SIMPATHIC
(irb):7: warning: previous definition of SIMPATHIC was here
(irb):8: warning: already initialized constant Object::RDFS
(irb):8: warning: previous definition of RDFS was here
drug lookup failed 4901d7c1b4e2aef6913d880bfc91240e
gene lookup failed 497258
gene lookup failed 60899174
gene lookup failed 3880727
gene lookup failed 31621
gene lookup failed 100275660
gene lookup failed 595111
gene lookup failed 41930
gene lookup failed 4326236
gene lookup failed 1280455
gene lookup failed 104889455
gene lookup failed 887303
gene lookup failed 3639254
gene lookup failed 14246
gene lookup failed 100009596
gene lookup failed 102552318
gene lookup failed 4338448
gene lookup failed 39965
gene lookup failed 100728057
gene lookup failed 125765250
gene lookup failed 125774680
gene lookup failed 17
gene lookup failed 816871
gene lookup failed 106478973
gene lookup failed 72448059
gene lookup failed 123168174
gene lookup failed 887216
gene

RDF quads written
